# 🌐 Dynamic Programming in GridWorld: Policy Evaluation, Iteration, and Value Iteration

This notebook implements and visualizes the classical Dynamic Programming algorithms used in Reinforcement Learning to solve Markov Decision Processes (MDPs). Under a known transition model, we implement:
1. **Iterative Policy Evaluation** (solving for the state-value function of a fixed policy)
2. **Policy Iteration** (alternating policy evaluation and greedy policy improvement)
3. **Value Iteration** (solving directly for the optimal value function using the Bellman Optimality operator)

---
## 1. Mathematical Formulations

### Bellman Expectation Equation (for Value Function $V^\pi(s)$)
$$V^\pi(s) = \sum_{a} \pi(a|s) \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V^\pi(s') \right]$$

### Bellman Optimality Equation (for Optimal Value Function $V^*(s)$)
$$V^*(s) = \max_{a} \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V^*(s') \right]$$


In [ ]:
import numpy as np

class GridWorld:
    def __init__(self):
        self.rows = 4
        self.cols = 4
        self.goal = (3, 3)
        self.actions = {
            'up': (-1, 0),
            'down': (1, 0),
            'left': (0, -1),
            'right': (0, 1)
        }
        
    def step(self, state, action):
        if state == self.goal:
            return state, 0, True
            
        current_row, current_col = state
        delta_row, delta_col = self.actions[action]
        
        new_row = current_row + delta_row
        new_col = current_col + delta_col
        
        # Boundary handling
        if new_row < 0 or new_row >= self.rows:
            new_row = current_row
        if new_col < 0 or new_col >= self.cols:
            new_col = current_col
            
        next_state = (new_row, new_col)
        reward = 1.0 if next_state == self.goal else 0.0
        done = True if next_state == self.goal else False
        
        return next_state, reward, done

env = GridWorld()


## 2. Iterative Policy Evaluation

Policy evaluation calculates the state-value function $V^\pi$ for a fixed policy $\pi$.
We iterate through all states, updating the state value based on the expected values of its next states:
$$V_{k+1}(s) \leftarrow \sum_{a} \pi(a|s) \sum_{s', r} P(s', r | s, a) [r + \gamma V_k(s')]$$


In [ ]:
def print_value_grid(V, title="Value Table"):
    print(f"\n=== {title} ===")
    grid = np.zeros((4, 4))
    for r in range(4):
        for c in range(4):
            grid[r, c] = V[(r, c)]
    print(np.round(grid, 4))

def policy_evaluation(env, policy, gamma=0.9, theta=1e-5):
    # Initialize value function
    V = { (r, c): 0.0 for r in range(env.rows) for c in range(env.cols) }
    
    while True:
        delta = 0
        for r in range(env.rows):
            for c in range(env.cols):
                state = (r, c)
                if state == env.goal:
                    continue
                    
                old_v = V[state]
                new_v = 0
                action_probs = policy[state]
                
                for action, prob in action_probs.items():
                    next_state, reward, done = env.step(state, action)
                    new_v += prob * (reward + gamma * V[next_state])
                    
                V[state] = new_v
                delta = max(delta, abs(old_v - new_v))
                
        if delta < theta:
            break
    return V

# Create uniform random policy
uniform_policy = {}
for r in range(env.rows):
    for c in range(env.cols):
        state = (r, c)
        if state != env.goal:
            uniform_policy[state] = {'up': 0.25, 'down': 0.25, 'left': 0.25, 'right': 0.25}

V_eval = policy_evaluation(env, uniform_policy)
print_value_grid(V_eval, "Values under Uniform Policy")


## 3. Policy Iteration

Policy Iteration guarantees convergence to the optimal policy by alternating:
1. **Policy Evaluation**: Compute $V^\pi$ for current policy $\pi$.
2. **Policy Improvement**: Update policy $\pi$ greedily with respect to $V^\pi$:
   $$\pi'(s) \leftarrow \arg\max_{a} \sum_{s', r} P(s', r | s, a) [r + \gamma V^\pi(s')]$$


In [ ]:
def policy_iteration(env, gamma=0.9, theta=1e-5):
    # Start with arbitrary policy (e.g. always move right)
    policy = {}
    for r in range(env.rows):
        for c in range(env.cols):
            if (r, c) != env.goal:
                policy[(r, c)] = {'up': 0.0, 'down': 0.0, 'left': 0.0, 'right': 1.0} # deterministically right
                
    V = { (r, c): 0.0 for r in range(env.rows) for c in range(env.cols) }
    iteration = 0
    
    while True:
        iteration += 1
        # Step 1: Policy Evaluation
        V = policy_evaluation(env, policy, gamma, theta)
        
        # Step 2: Policy Improvement
        policy_stable = True
        for r in range(env.rows):
            for c in range(env.cols):
                state = (r, c)
                if state == env.goal:
                    continue
                    
                old_action = max(policy[state], key=policy[state].get)
                
                # Greedy improvement
                best_action = None
                best_val = float('-inf')
                for action in env.actions:
                    next_state, reward, done = env.step(state, action)
                    val = reward + gamma * V[next_state]
                    if val > best_val:
                        best_val = val
                        best_action = action
                        
                # Update policy deterministically
                new_action_probs = {act: 0.0 for act in env.actions}
                new_action_probs[best_action] = 1.0
                policy[state] = new_action_probs
                
                if best_action != old_action:
                    policy_stable = False
                    
        print(f"Policy Iteration {iteration} complete.")
        if policy_stable:
            break
            
    return policy, V

opt_policy_pi, opt_v_pi = policy_iteration(env)
print_value_grid(opt_v_pi, "Optimal Value Function (Policy Iteration)")

print("\n=== Optimal Policy (PI) ===")
for r in range(env.rows):
    row_str = []
    for c in range(env.cols):
        state = (r, c)
        if state == env.goal:
            row_str.append(" GOAL ")
        else:
            act = max(opt_policy_pi[state], key=opt_policy_pi[state].get)
            row_str.append(f"{act:^6}")
    print(" | ".join(row_str))


## 4. Value Iteration

Value Iteration collapses evaluation and improvement into a single step by updating the values directly using the Bellman Optimality Equation:
$$V_{k+1}(s) \leftarrow \max_{a} \sum_{s', r} P(s', r | s, a) [r + \gamma V_k(s')]$$


In [ ]:
def value_iteration(env, gamma=0.9, theta=1e-5):
    V = { (r, c): 0.0 for r in range(env.rows) for c in range(env.cols) }
    iteration = 0
    
    while True:
        iteration += 1
        delta = 0
        for r in range(env.rows):
            for c in range(env.cols):
                state = (r, c)
                if state == env.goal:
                    continue
                    
                old_v = V[state]
                
                # Find max value over all actions
                best_val = float('-inf')
                for action in env.actions:
                    next_state, reward, done = env.step(state, action)
                    val = reward + gamma * V[next_state]
                    best_val = max(best_val, val)
                    
                V[state] = best_val
                delta = max(delta, abs(old_v - best_val))
                
        if delta < theta:
            print(f"Value Iteration converged in {iteration} steps.")
            break
            
    # Extract policy greedily from converged values
    policy = {}
    for r in range(env.rows):
        for c in range(env.cols):
            state = (r, c)
            if state == env.goal:
                continue
                
            best_action = None
            best_val = float('-inf')
            for action in env.actions:
                next_state, reward, done = env.step(state, action)
                val = reward + gamma * V[next_state]
                if val > best_val:
                    best_val = val
                    best_action = action
            policy[state] = best_action
            
    return policy, V

opt_policy_vi, opt_v_vi = value_iteration(env)
print_value_grid(opt_v_vi, "Optimal Value Function (Value Iteration)")

print("\n=== Optimal Policy (VI) ===")
for r in range(env.rows):
    row_str = []
    for c in range(env.cols):
        state = (r, c)
        if state == env.goal:
            row_str.append(" GOAL ")
        else:
            act = opt_policy_vi[state]
            row_str.append(f"{act:^6}")
    print(" | ".join(row_str))


## 5. Comparison & Conclusion

- **Policy Iteration** takes fewer outer iterations (usually 2-3 iterations) but requires running Policy Evaluation to convergence within each iteration.
- **Value Iteration** runs a single sweep of values per iteration and converges in more sweeps (around 7 steps here), but requires no inner evaluations.
- Both methods converge to the **exact same optimal value function and optimal policy**.
